In [ ]:
import numpy as np
import matplotlib
from millab.src.builder import create_model
import os
import h5py
import torch
from trident import OpenSlideWSI, visualize_heatmap
import torch.nn.functional as F


matplotlib.use("Agg")
import matplotlib.pyplot as plt


# Run inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
slide_id = "004"
artifacts_dir = "artifacts/testung"
job_dir = "./heatmaps/moe_no_cpls"
fold_num = 0
trial_num = 1
patch_features_path = f"features/features_conch_v15_HUN/{slide_id}.h5"
slide_path = f"D:\\2026_CRC\\slides\\{slide_id}.mrxs"


# Model loading
model_path = os.path.join(artifacts_dir, f"trial_{trial_num}", "models", f"fold_{fold_num}.pt")



model = create_model('abmil.base_mammoth.conch_v15', num_classes=4, return_attention=True).to(device)
model.load_state_dict(torch.load(model_path, map_location=torch.device(device)), strict=False)
model.to(device)


slide = OpenSlideWSI(slide_path, lazy_init=False)
with h5py.File(patch_features_path, 'r') as f:
    coords = f['coords'][:]
    patch_features = f['features'][:]
    coords_attrs = dict(f['coords'].attrs)

batch = {'features': torch.from_numpy(patch_features).float().to(device).unsqueeze(0)}
model.eval()


with torch.no_grad(): # Saves memory during inference

    log = model(batch['features'])

    # Stage 2: attention over the P = E*S slots
    A = F.softmax(log[1]["attention"], dim=-1).squeeze(1)  # [B, P]  (K=1 -> squeeze)

    # Stage 1: router weights, tile -> slot. Fold heads and (E,S) -> P.
    dispatch = log[1]["moe_weights"]  # [B, N, E, H, S], softmax over N
    B, N, E, Hh, S = dispatch.shape
    D = dispatch.mean(dim=3)  # [B, N, E, S]   average over heads
    D = D.reshape(B, N, E * S)  # [B, N, P], ordering p = e*S + s

    # Rollout
    patch_attn = torch.einsum("bp,bnp->bn", A, D)  # [B, N]

    patch_attn = patch_attn / patch_attn.amax(dim=-1, keepdim=True).clamp_min(1e-12)


# Extract the numpy array
raw_scores = patch_attn.cpu().numpy().squeeze()


# Reduce to 1D if we have multiple heads
if raw_scores.ndim >= 2:
    # Check which axis represents the heads (the axis with size 2)
    head_axis = 1 if raw_scores.shape[0] == 2 else 0

    # Option A: Average the attention scores across the heads (Recommended)
    scores_1d = np.mean(raw_scores, axis=head_axis)

    # Option B: Or, just select the first head
    # scores_1d = raw_scores[0, :] if head_axis == 0 else raw_scores[:, 0]
else:
    scores_1d = raw_scores

# f. generate heatmap
heatmap_save_path = visualize_heatmap(
    wsi=slide,
    scores=scores_1d, # Pass the corrected 1D array here
    coords=coords,
    vis_level=None,
    vis_mag=5,
    patch_size_level0=coords_attrs['patch_size_level0'],
    normalize=True,
    num_top_patches_to_save=10,
    output_dir=job_dir,
    filename=slide_id+".tif",
)
